In [1]:
from transformers import pipeline
import torch

# Syntaxe Transformers 4.44+ compatible Pylance
ner_pipeline = pipeline("ner", 
                       model="dbmdz/bert-large-cased-finetuned-conll03-english",
                       aggregation_strategy="simple"
                       )

summarizer = pipeline("summarization", 
                     model="facebook/bart-large-cnn")

def ocr_simule(image):
    return """
FACTURE #12345 | Date: 15/12/2025 | Client: Jean Dupont
Livre Python: 25€ | Formation IA: 150€ | Total: 175€
TechFormations SARL"""

# Reste identique...
def traiter_document_reel(image):
    if image is None: 
        return "Pas d'image", "Chargez une image", []
    texte = ocr_simule(image)
    entites = ner_pipeline(texte)[:5]
    resume = summarizer(texte[:1000], max_length=100, min_length=30, do_sample=False)[0]['summary_text']
    return texte, resume, entites


c:\Users\julie\OneDrive\Documents\Briefs\Projet_groupe\Summarize_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set t

In [ ]:
from transformers import pipeline

# Fonctionne malgré les warnings Pylance
ner_pipeline = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", aggregation_strategy="simple")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Test direct (copiez/exécutez)
texte_test = """hapitre 3
Passage entre systèmes de
coordonnées
Les différents systèmes de coordonnées ne sont pas hermétiques entre eux, et
il est nécessaire de pouvoir passer de l’un à l’autre. Il existe essentiellement
2 façons de convertir les coordonnées :
• utilisation de la trigonométrie sphérique
• utilisation des matrices de rotation
3.1
Trigonométrie sphérique
Un triangle sphérique est un triangle dessiné sur une sphère. Ses côtés ne
sont plus des segments mais des arcs de grands cercles de cette sphère. Les
règles habituelles de la trigonométrie euclidienne ne sont plus applicables;
par exemple, la somme des angles d’un triangle sphérique est supérieure à
180°.
On va considérer le rayon de cette sphère, de centre O comme unité.
3.1.1 Groupe de Gauss
On considère un triangle sphérique de sommets A, B, C. L'angle Ā représente
l’angle entre les tangentes au point A aux arcs de grands cercles AB et AC.
On définit de même les angles B et . Les arcs AB, AC et BC sont respecti-
vement interceptés par les angles au centre c, b et a. Finalement, on projette
orthogonalement les points B en C sur l'axe (OA) en respectivement B' et"""
print(ner_pipeline(texte_test))  # ✅ Fonctionne
print(summarizer(texte_test))    # ✅ Fonctionne


Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu


[{'entity_group': 'MISC', 'score': np.float32(0.65919334), 'word': '##ienne', 'start': 553, 'end': 558}, {'entity_group': 'ORG', 'score': np.float32(0.94774276), 'word': 'Groupe de Gauss', 'start': 739, 'end': 754}, {'entity_group': 'ORG', 'score': np.float32(0.7874), 'word': 'AB', 'start': 953, 'end': 955}, {'entity_group': 'ORG', 'score': np.float32(0.8868856), 'word': 'AC', 'start': 957, 'end': 959}, {'entity_group': 'ORG', 'score': np.float32(0.74844706), 'word': 'BC', 'start': 963, 'end': 965}]
[{'summary_text': 'Trigonométrie sphérique est un triangle dessiné sur une sphère. Ses côtés ne sont plus des segments mais des arcs de grands cercles. Les règles habituelles de la trigonometrie euclidienne ne sons plus applicables; par exemple, la somme des angles d’un triangle sphÉrique est supérieure à 180°.'}]


In [19]:
import re
from transformers import pipeline


# ---------------------------------------------------------
# Chargement du modèle (UNE SEULE FOIS)
# ---------------------------------------------------------

print("[INFO] Loading summarization model...")

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

print("[INFO] Model loaded.")


# ---------------------------------------------------------
# Nettoyage générique du texte
# ---------------------------------------------------------

def clean_text(text: str) -> str:
    text = re.sub(r"[€·÷≤]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ---------------------------------------------------------
# Détection automatique de structure
# ---------------------------------------------------------

def detect_sections(text: str) -> list[str]:
    """
    Tente de détecter des sections.
    Si échec → fallback sur découpage par taille.
    """

    # Motifs courants de titres
    patterns = [
        r"\n\d+\.\d+(?:\.\d+)?",     # 3.1 / 3.1.1
        r"\n\d+\s",                  # 1 Title
        r"\n[IVX]+\)",               # I) II)
        r"\n[A-Z][^\n]{0,40}\n"      # TITRE EN MAJUSCULES
    ]

    for pattern in patterns:
        splits = re.split(pattern, text)
        sections = [s.strip() for s in splits if len(s.strip()) > 200]
        if len(sections) > 1:
            return sections

    return []


# ---------------------------------------------------------
# Fallback : découpage intelligent par taille
# ---------------------------------------------------------

def chunk_text(text: str, max_chars: int = 900) -> list[str]:
    chunks = []
    current = ""

    for sentence in re.split(r"(?<=[.!?])\s+", text):
        if len(current) + len(sentence) <= max_chars:
            current += " " + sentence
        else:
            chunks.append(current.strip())
            current = sentence

    if current:
        chunks.append(current.strip())

    return chunks


# ---------------------------------------------------------
# Résumé d’un bloc
# ---------------------------------------------------------

def summarize_block(text: str,
                    max_length: int = 120,
                    min_length: int = 40) -> str:
    result = summarizer(
        text,
        max_length=max_length,
        min_length=min_length,
        do_sample=False
    )
    return result[0]["summary_text"]


# ---------------------------------------------------------
# Résumé générique hiérarchique
# ---------------------------------------------------------

def adaptive_summarization(text: str) -> str:
    cleaned = clean_text(text)

    # 1️⃣ Tentative de découpage par sections
    sections = detect_sections(cleaned)

    # 2️⃣ Fallback si aucune structure détectée
    if not sections:
        sections = chunk_text(cleaned)

    print(f"[INFO] {len(sections)} text blocks detected.")

    # Résumé de chaque bloc
    block_summaries = []
    for i, block in enumerate(sections, start=1):
        print(f"[INFO] Summarizing block {i}...")
        summary = summarize_block(block)
        block_summaries.append(summary)

    # Résumé final
    combined = " ".join(block_summaries)

    print("[INFO] Generating final summary...")

    final_summary = summarize_block(
        combined,
        max_length=160,
        min_length=60
    )

    return final_summary


# ---------------------------------------------------------
# TEST
# ---------------------------------------------------------

if __name__ == "__main__":

    texte = """
    hapitre 3
Passage entre systèmes de
coordonnées
Les différents systèmes de coordonnées ne sont pas hermétiques entre eux, et
il est nécessaire de pouvoir passer de l’un à l’autre. Il existe essentiellement
2 façons de convertir les coordonnées :
• utilisation de la trigonométrie sphérique
• utilisation des matrices de rotation
3.1
Trigonométrie sphérique
Un triangle sphérique est un triangle dessiné sur une sphère. Ses côtés ne
sont plus des segments mais des arcs de grands cercles de cette sphère. Les
règles habituelles de la trigonométrie euclidienne ne sont plus applicables;
par exemple, la somme des angles d’un triangle sphérique est supérieure à
180°.
On va considérer le rayon de cette sphère, de centre O comme unité.
3.1.1 Groupe de Gauss
On considère un triangle sphérique de sommets A, B, C. L'angle Ā représente
l’angle entre les tangentes au point A aux arcs de grands cercles AB et AC.
On définit de même les angles B et . Les arcs AB, AC et BC sont respecti-
vement interceptés par les angles au centre c, b et a. Finalement, on projette
orthogonalement les points B en C sur l'axe (OA) en respectivement B' et
    """

    summary = adaptive_summarization(texte)

    print("\n--- RÉSUMÉ FINAL ---\n")
    print(summary)


[INFO] Loading summarization model...


Device set to use cpu


[INFO] Model loaded.
[INFO] 2 text blocks detected.
[INFO] Summarizing block 1...


Your max_length is set to 120, but your input_length is only 103. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=51)


[INFO] Summarizing block 2...


Your max_length is set to 160, but your input_length is only 136. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=68)


[INFO] Generating final summary...

--- RÉSUMÉ FINAL ---

L'angle Ā représente l’angle entre les tangentes au point A aux arcs de grands cercles AB and AC. Les arcs AB, AC et BC sont respecti- vement interceptés par les angles au centre c, b and a.


In [22]:
import re
from transformers import pipeline

# Chargement du modèle
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")


def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def balanced_chunking(text: str, max_chars: int = 800) -> list[str]:
    """
    Découpe le texte en blocs équilibrés,
    en conservant le début, le milieu et la fin.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) <= max_chars:
            current += " " + sentence
        else:
            chunks.append(current.strip())
            current = sentence

    if current:
        chunks.append(current.strip())

    return chunks


def summarize_block(text: str) -> str:
    result = summarizer(
        text,
        max_length=250,
        min_length=40,
        do_sample=False
    )
    return result[0]["summary_text"]


def global_summary(text: str) -> str:
    cleaned = clean_text(text)

    chunks = balanced_chunking(cleaned)

    print(f"[INFO] {len(chunks)} chunks created.")

    block_summaries = []
    for i, chunk in enumerate(chunks, 1):
        print(f"[INFO] Summarizing chunk {i}...")
        block_summaries.append(summarize_block(chunk))

    combined = " ".join(block_summaries)

    print("[INFO] Generating final summary...")

    final = summarize_block(
        combined
    )

    return final

texte = """
    hapitre 3
Passage entre systèmes de
coordonnées
Les différents systèmes de coordonnées ne sont pas hermétiques entre eux, et
il est nécessaire de pouvoir passer de l’un à l’autre. Il existe essentiellement
2 façons de convertir les coordonnées :
• utilisation de la trigonométrie sphérique
• utilisation des matrices de rotation
3.1
Trigonométrie sphérique
Un triangle sphérique est un triangle dessiné sur une sphère. Ses côtés ne
sont plus des segments mais des arcs de grands cercles de cette sphère. Les
règles habituelles de la trigonométrie euclidienne ne sont plus applicables;
par exemple, la somme des angles d’un triangle sphérique est supérieure à
180°.
On va considérer le rayon de cette sphère, de centre O comme unité.
3.1.1 Groupe de Gauss
On considère un triangle sphérique de sommets A, B, C. L'angle Ā représente
l’angle entre les tangentes au point A aux arcs de grands cercles AB et AC.
On définit de même les angles B et . Les arcs AB, AC et BC sont respecti-
vement interceptés par les angles au centre c, b et a. Finalement, on projette
orthogonalement les points B en C sur l'axe (OA) en respectivement B' et
    """
summary = summarizer(
    texte,
    max_length=300,      # ↑ longueur max
    min_length=150,      # ↑ longueur min
    length_penalty=2.0,  # favorise les résumés plus longs
    num_beams=4,         # améliore la qualité
    do_sample=False
)

print(summary[0]["summary_text"])



Device set to use cpu


Trigonométrie sphérique est un triangle dessiné sur une sphère. Les différents systèmes de coordonnées ne sont pas hermétiques entre eux, et nécessaire de pouvoir passer de l’un à l'autre. Il existe essentiellement two façons de convertir les coordonnsées : utilisation of the trigonometrie and of the matrices de rotation. Theory of trigonométries is based on trigonométrie euclidienne, a.k.a. ‘theory of angles’, ‘synthesis of triangles’ and ‘triangle theory’
